# Feature Engineering and Feature Selection

## Setup

Run the following code to import the necessary libraries/modules.

In [9]:
## Setup
import pandas as pd
import numpy as np
import os
import math
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import itertools
import sys
import pathlib
import datetime as dt
import notebook_file_utilities.auto_add_project_root as proroot

# For reading in modules from src and reading/saving files
project_root = proroot.auto_add_project_root()

from src.pipeline.preprocessing import preprocessing
from src.cleaning.merged_tornado_indicator import create_tornado_indicator
import src.cleaning.na_imputer as na_imp
import src.cleaning.drop_features as dfeat
import Data.metadata.cleaned_feature_info as featinfo

The code below reads in processed data, imputes nan values, drops quality code features, and drops duplicates as in the eda notebook.

In [3]:
data = create_tornado_indicator(time_window=1,val_radius=50)

/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:93: DtypeWarning: Columns (7,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,40,41,42,43,44,45,46,47,50,51,52,56,57,58,59,60,65,68,69,70,71,76,79,80,81,82,83,90,91,92,93,94,95,96,97,98,99,102,104,105,106,110,111,113,120,121,122,125) have mixed types. Specify dtype option on import or set low_memory=False.
  station_dfs=[pd.read_csv(os.path.join(station_dir,file)).copy() for file in station_csv_files]
/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:93: DtypeWarning: Columns (39,40,41,42,43,47,48,52,53,54,55,57,58,59,60,61,65,70,71,76,77,88,89,106,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  station_dfs=[pd.read_csv(os.path.join(station_dir,file)).copy() for file in station_csv_files]
/Users/taylormurray/Documents/GitHub/fall-2025-predicting-tornadoes/src/cleaning/clean_stations.py:93: Dt

## Necessary Feature Scaling 

There are a few features that are scaled by 10 (or some other value) from the station data we collected. These are summarized in [Data/metadata/cleaned_feature_info.py](<../Data/metadata/cleaned_feature_info.py>). For convenience, we copy all (potentially) scaled features here:

* CIG- Sky Condition Observation- Ceiling Height Dimension : scaled by 1
* DEW- Air Temperature Observation- Dew Point Temperature : scaled by 10
* TMP- Air Temperature Observation- Air Temperature : scaled by 10
* MA1- Atmospheric Pressure Observation- Altimeter Setting Rate : scaled by 10
* MA1- Atmospheric Pressure Observation- Station Pressure Rate : scaled by 10
* SLP- Atmospheric Pressure Observation- Sea Level Pressure : scaled by 10
* VIS- Visibility Observation- Distance Dimension : scaled by 1
* WND- Wind Observation- Direction Angle : scaled by 1
* WND- Wind Observation- Speed Rate : scaled by 10


The following code applies the necessary scaling to convert these values to their proper scale and displays the original column side-by-side for quick verification. Moreover we replace the original column with the appropriately its scaled version

## Temporal Features (Pre Train-Test Split)


| Feature                  | Description                          | Type                 | Why it’s useful                                  |
| ------------------------ | ------------------------------------ | -------------------- | ------------------------------------------------ |
| `year`                   | Year of observation                  | numeric              | allows chronological splits, long-term trends    |
| `sin_monthofyear`  and `cos_monthofyear`              |  given month of observation as a number from 1--12, these features have values: $$\sin(month*2\pi/12)$$ and $$\cos(month*2\pi/12)$$ respectively.| cyclic | captures seasonal tornado cycles                 |
| `sin_dayofyear`  and `cos_dayofyear`          | given day of observation as a number from 1--365, these features have values: $$\sin(day*2\pi/365)$$ and $$\cos(day*2\pi/365)$$ respectively.      |  cyclic     | finer seasonal resolution                        |
|        `sin_hourofday`  and `cos_hourofday`                 | given time of observation as a number from 0--23, these features have values: $$\sin(hour*2\pi/24)$$ and $$\cos(hour*2\pi/24)$$ respectively.                 |  cyclic | diurnal cycle of convection                      |



## Temporal Features (Post Train-Test Split)

| Feature                    | Description                               | Type | Why it's useful rule                                                        |
| -------------------------- | ----------------------------------------- | ------------------- |---- |
| `lag_temp_1h`              | temperature 1 hour before current observation                | numeric                  | Tracks temperature changes|



## Feature Engineering (Pre Train-Test Split)

Our next step is Feature Engineering.

| New Feature                     | Definition                                                   | Motivation                                                                        |
| ------------------------------- | ------------------------------------------------------------ | --------------------------------------------------------------------------------- |
| **Temperature–Dewpoint Spread** | `TMP - DEW`                                                  | This is the wet-bulb depression. Together with dewpoint this is often a good indicator for storm occurrence.     |                                     |





## Scaling and Transformation Strategies


| Model Type                                     | Scaling Plan                                                         |
| ---------------------------------------------- | -------------------------------------------------------------------- |
| **Tree-based models (Decision Tree, XGBoost)** | No scaling needed; use raw or log-transformed values where skewed.   |



## Drop Features

| Category                          | Features to Drop                                                                                                                                      | Rationale                            |
| --------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------ |
| **Pressure redundancy**           | `MA1 Altimeter Rate`, `SLP`                                                                                                     | > 0.99 correlation                   |
| **Thermodynamic redundancy**      | One of `TMP` or `DEW`                                                                                                                                 | +0.84 correlation                    |
| **Geospatial/labeling**           | `STATION_LAT`, `STATION_LON`, `TORNADO_BEGIN_DATE_TIME`, `TORNADO_END_DATE_TIME`,`TORNADO_BEGIN_LAT`, `TORNADO_BEGIN_LON`, `TORNADO_END_LAT`, `TORNADO_END_LON`, `TORNADO_INITIAL_DISTANCE_FROM_STATION`, `TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_50_km` | Used to derive label (avoid data leakage with the TORNADO features)                |
| **Censored / non-informative**    | `VIS`, `CIG`                                                                                                                                          | Maxed-out values, non-discriminative |
| **Weak / unrepresentative winds** | `WND-Direction Angle`, `WND-Type Code`                                                                                                                | Local-only; low predictive power     |

